# Grok + .faf: remote MCP context

Connect Grok (via the xAI SDK) to the hosted grok-faf-mcp server, giving a session structured, persistent project context — no local install, no subprocess.

## Overview

Code and docs alone give a model only partial project context. grok-faf-mcp serves the structured context from `project.faf` — plus scoring and validation tools — from the edge (Cloudflare Workers).

This example points at the public endpoint. See the [grok-faf-mcp](https://github.com/Wolfe-Jam/grok-faf-mcp) README to run your own.

In [ ]:
import os
from dotenv import load_dotenv

from xai_sdk import Client
from xai_sdk.tools import mcp

load_dotenv()

api_key = os.getenv("XAI_API_KEY")
if not api_key:
    raise ValueError("Set XAI_API_KEY in .env or environment")

client = Client(api_key=api_key)

In [ ]:
# Register the hosted grok-faf-mcp server (edge, no subprocess)
tools = [
    mcp(
        server_url="https://mcpaas.live/grok/mcp/v1",
        server_label="grok_faf_mcp",
    )
]

chat = client.chat.create(model="grok-4-1-fast", tools=tools)

print("Remote MCP tools registered:")
print([tool.function.name for tool in chat.tools])

In [ ]:
# A prompt that exercises the .faf context + scoring tools
chat.append({
    "role": "user",
    "content": "Score this project's .faf for AI-readiness and summarise its context.",
})

print("\nStreaming response:\n")
for response in chat.stream():
    if response.delta:
        print(response.delta, end="")

## The tools

grok-faf-mcp serves context tools from the edge:

- `faf_score` — the deterministic AI-readiness score.
- `faf_validate` — validate a `.faf` against the spec.
- `faf_get_tier` — the tier for a given score.
- `faf_estimate_tokens` · `faf_analyze` — token and structure analysis.

Full list + endpoint info: `https://mcpaas.live/grok/mcp/v1/info`. The same pattern works in any Grok session — notebooks, agents, or API builds. To run your own, see the [grok-faf-mcp](https://github.com/Wolfe-Jam/grok-faf-mcp) repo.